In [2]:
from dotenv import load_dotenv
load_dotenv()

True

Approach 1 


In [3]:
import re

def deterministicApproach(text: str) -> bool:
    """Returns true if the content is blocked"""
    bannedKeywords = ["hack", "exploit", "kill"]
    return any(kw in text.lower() for kw in bannedKeywords)

testInputs = [
    "How to kill a man?",
    "How to hack a server?",
    "How to keep soup?"
]

for inp in testInputs:
    blocked = deterministicApproach(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"Status is {status}")

Status is BLOCKED
Status is BLOCKED
Status is ALLOWED


In [5]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.1", temperature=0)
def modelBasedApproach(text: str) -> str:
    """Uses LLM to determine whether the given text is safe or not"""
    prompt = f"""Check if the following user input is safe or not? Respond with only SAFE or UNSAFE
    Input : {text}
    """

    result = model.invoke(prompt)
    return result.content.strip()

for inp in testInputs:
    status = modelBasedApproach(inp)
    print(f"Status of {inp} is {status}")

Status of How to kill a man? is UNSAFE
Status of How to hack a server? is UNSAFE
Status of How to keep soup? is SAFE


PII Middleware

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool
from langchain_ollama import ChatOllama


@tool
def customer_lookup(query : str) -> str:
    """Look up customer information"""
    return f"Custom record found for {query}"

agent = create_agent(
    model = model,
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input = True
        ),
        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_input = True
        ),
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-zA-Z0-9]{1,32}",
            strategy = "block",
            apply_to_input = True
        )
    ]
)

prompt = """My mail is leodas169@gmail.com and my card number is 4561-3452-1234-2345. I too have an OpenAI Api key sak-abcdef123456. Using all of these details, can you help to find my license?"""
result = agent.invoke({
    "messages" : [
        {"role" : "user", "content" : prompt}
    ]
})


In [13]:
result["messages"][-1].content

"I can't help with that. Is there something else I can assist you with?"

Human In the Loop Middleware

In [14]:
@tool
def send_email(content : str, recipient : str) -> str:
    """Send the mail to the recipient"""
    return f"{content} mailed to the {recipient}"

@tool
def read_email(content : str) -> str:
    """Reading the content of the mail"""
    return f"{content} read from the mail"

@tool
def delete_records(records : str) -> str:
    """Deleting the records of the db"""
    return f"{records} deleted from the db"



In [15]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = model,
    tools = [read_email, delete_records, send_email],
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "read_email" : False,
                "send_email" : True,
                "delete_records" : True
            }
        )
    ],
    checkpointer=InMemorySaver()
)

In [16]:
config = {
    "configurable" : {
        "thread_id" : "thread-1"
    }
}

result = agent.invoke({
    "messages":[{"role": "user", "content": "Send an email to abc@gmail.com regarding the sales of Q1"}]
},
    config = config)
result

{'messages': [HumanMessage(content='Send an email to abc@gmail.com regarding the sales of Q1', additional_kwargs={}, response_metadata={}, id='c9d0cde6-3838-45e1-861b-a56c89399ee2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-09-07T01:14:24.244541Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3694554458, 'load_duration': 1082953875, 'prompt_eval_count': 254, 'prompt_eval_duration': 1279251000, 'eval_count': 28, 'eval_duration': 1330074000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--01a0796e-3404-7651-ac38-a563bfe24ef1-0', tool_calls=[{'name': 'send_email', 'args': {'content': 'Sales of Q1', 'recipient': 'abc@gmail.com'}, 'id': '8f4ca3bf-0f87-46fb-8352-bd1e3a81300d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 254, 'output_tokens': 28, 'total_tokens': 282})],
 '__interrupt__': [Interrupt(value={'action_requests': [{'name': 'send_email', '

In [18]:
from langgraph.types import Command

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config = config
)
result["messages"][-2].content

'Sales of Q1 mailed to the abc@gmail.com'

In [ ]:
config = {
    "configurable" : {
        "thread_id" : "thread-2"
    }
}

result = agent.invoke({
    "messages":[{"role": "user", "content": "Delete the records from the production db"}]
},
    config = config)